In [1]:
import pandas as pd

df = pd.read_csv('salaries.csv')
df.head()

,company,job,degree,salary_more_then_100k
0,google,sales executive,bachelors,0
1,google,sales executive,masters,0
2,google,business manager,bachelors,1
3,google,business manager,masters,1
4,google,computer programmer,bachelors,0


### Feature engineering section: Label and One hot encoding

In [ ]:
# encoding the degree column, Type of encoding is label encoding as there are only two categories in degree column
# ordinal encoding is used when there is a natural order between the categories, in this case, we can say that masters degree is higher than bachelors degree, so we can assign 1 to bachelors and 2 to masters
# Example of ordinal encoding is when we have a column called "education level" with categories "high school", "bachelors", "masters", "phd", we can assign 1 to high school, 2 to bachelors, 3 to masters and 4 to phd
df['degree_number'] = df['degree'].map({'bachelors':1,'masters':2})
df.head()

,company,job,degree,salary_more_then_100k,degree_number
0,google,sales executive,bachelors,0,1
1,google,sales executive,masters,0,2
2,google,business manager,bachelors,1,1
3,google,business manager,masters,1,2
4,google,computer programmer,bachelors,0,1


In [ ]:
df.drop('degree',axis="columns",inplace=True)


KeyError: "['degree'] not found in axis"

In [5]:
df.head()

,company,job,salary_more_then_100k,degree_number
0,google,sales executive,0,1
1,google,sales executive,0,2
2,google,business manager,1,1
3,google,business manager,1,2
4,google,computer programmer,0,1


In [8]:
# One hot encoding for company and job columns
# nominal encoding is used when there is no natural order between the categories, in this case, 
# we can say that there is no natural order between the companies and jobs, so we can use one hot encoding to create new columns for each category in company and job columns
df_encoded = pd.get_dummies(df,columns=['company','job'], drop_first=True)
df_encoded.head()

,salary_more_then_100k,degree_number,company_facebook,company_google,job_computer programmer,job_sales executive
0,0,1,False,True,False,True
1,0,2,False,True,False,True
2,1,1,False,True,False,False
3,1,2,False,True,False,False
4,0,1,False,True,True,False


### Model training

In [10]:
from sklearn.tree import DecisionTreeClassifier

X=df_encoded.drop('salary_more_then_100k',axis='columns')
y=df_encoded['salary_more_then_100k']
model = DecisionTreeClassifier()
model.fit(X, y)


DecisionTreeClassifier()

In [11]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X)
print(classification_report(y,y_pred))


              precision    recall  f1-score   support

           0       1.00      1.00      1.00         6
           1       1.00      1.00      1.00        10

    accuracy                           1.00        16
   macro avg       1.00      1.00      1.00        16
weighted avg       1.00      1.00      1.00        16



In [13]:
from sklearn.tree import export_text
tree_rules = export_text(model, feature_names=list(X.columns))
print(tree_rules)

print(list(X.columns))

|--- company_facebook <= 0.50
|   |--- job_sales executive <= 0.50
|   |   |--- degree_number <= 1.50
|   |   |   |--- company_google <= 0.50
|   |   |   |   |--- class: 0
|   |   |   |--- company_google >  0.50
|   |   |   |   |--- job_computer programmer <= 0.50
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- job_computer programmer >  0.50
|   |   |   |   |   |--- class: 0
|   |   |--- degree_number >  1.50
|   |   |   |--- class: 1
|   |--- job_sales executive >  0.50
|   |   |--- class: 0
|--- company_facebook >  0.50
|   |--- class: 1

['degree_number', 'company_facebook', 'company_google', 'job_computer programmer', 'job_sales executive']


In [14]:
model.predict([
    [1,1,0,1,0],
    [1,0,1,0,1]
])

c:\Office_Data\Prsonal data\Prsonal data\Resume\myrepo\AI_Solutions_using_Azure_AI\PracticeCode\.venv\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but DecisionTreeClassifier was fitted with feature names
  warnings.warn(


array([1, 0])